In [51]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as pl
import seaborn as sns
from tqdm.auto import tqdm

In [52]:
DIR = Path("/g/huber/users/peidli/data/SYMPHONY/SYM10RNAseq")

In [53]:
sample_list = list(pd.read_csv(DIR / "sample_list.txt", index_col=0, header=None).index)

In [ ]:
tabs = {}
for sample in tqdm(sample_list):
    d = DIR / f'{sample}/RNA/counts/'
    if d.exists():
        tab = pd.read_csv(d / f"{sample}.converted_counts.tsv.gz", compression='gzip', sep='\t', header=0, skiprows=6, index_col=(0,1,2,3,4))
        tabs[sample] = tab['count']


  0%|          | 0/345 [00:00<?, ?it/s]

In [122]:
df = pd.concat(tabs, axis=1)
df = df[~df.index.get_level_values(1).duplicated(keep='first')]
df.to_csv(DIR / "prelim_combined_counts.tsv", sep='\t', header=True)

In [69]:
df = pd.read_csv(DIR / "prelim_combined_counts.tsv", sep='\t', header=0, index_col=(0,1,2,3,4))

In [70]:
df

,,,,,75CAS_FBL_V1_R1,75CAS_FBL_V1_R2,75CAS_FBL_V1_R3,75CAS_FBL_V2_R1,75CAS_FBL_V2_R2,75CAS_FBL_V2_R3,75CAS_FBL_V3_R1,75CAS_FBL_V3_R2,75CAS_FBL_V3_R3,75CAS_FTL_V1_R1,...,99CAS_SNHG16_V2_R3,99CAS_VIM_V1_R1,99CAS_VIM_V1_R2,99CAS_VIM_V1_R3,99CAS_VIM_V2_R1,99CAS_VIM_V2_R2,99CAS_VIM_V2_R3,99CAS_VIM_V3_R1,99CAS_VIM_V3_R2,99CAS_VIM_V3_R3
#ensid,gene,biotype,chr,longest_isoform,,,,,,,,,,,,,,,,,,,,,
ENSG00000000003,TSPAN6,protein_coding,X,3796,765,661,725,736,725,558,681,656,726,740,...,512,1388,1192,994,1049,1064,1065,1465,1480,1385
ENSG00000000005,TNMD,protein_coding,X,1205,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
ENSG00000000419,DPM1,protein_coding,20,1161,1052,864,1047,1035,934,817,1026,1020,1058,738,...,672,1005,838,695,805,800,837,944,955,809
ENSG00000000457,SCYL3,protein_coding,1,6308,202,185,239,246,194,206,230,205,238,221,...,124,169,161,149,194,152,168,184,213,167
ENSG00000000460,C1orf112,protein_coding,1,4355,476,358,447,477,421,390,462,413,454,356,...,321,562,548,415,441,404,479,415,447,358
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ERCC-00164,ERCC-00164,unknown,ERCC-00164,1022,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
ERCC-00165,ERCC-00165,unknown,ERCC-00165,872,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
ERCC-00168,ERCC-00168,unknown,ERCC-00168,1024,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [118]:
df[df.index.get_level_values(1).duplicated(keep=False)].index.get_level_values(1).unique()

Index(['MATR3', 'CYB561D2', 'PDE11A', 'HERC3', 'RGS5', 'CCDC39', 'TMSB15B',
       'ELFN2', 'POLR2J3', 'ACTL10', 'SPATA13', 'GOLGA8M', 'Y_RNA',
       'Metazoa_SRP', 'U3', 'SNORA63', 'SNORD33', 'U8', 'SNORD81', 'SNORA73',
       'SNORA70', 'SNORA72', 'SNORD18', 'SNORA62', 'SNORD42', 'SNORA71', '7SK',
       'SNORD116', 'SLFN12L', 'DUXAP8', 'SNORA75', 'SNORA50A', 'SNORD63',
       'SNORD38B', 'SNORD115', 'SNORD30', 'POLR2J4', 'RAET1E-AS1',
       'DNAJC9-AS1', 'ZNF883', 'SFTA3', 'LINC00484', 'LINC00486', 'ANAPC1P2',
       'CLCA4-AS1', 'DGCR5', 'LINC01115', 'LINC01238', 'GABARAPL3',
       'PRICKLE2-AS1', 'BMS1P4', 'ALG1L9P', 'SNORD27', 'SNORA74', 'Vault',
       'KBTBD11-OT1', 'PINX1', 'ITFG2-AS1', 'SNORD39', 'RMRP',
       'ARMCX5-GPRASP2', 'U7', 'U1', 'U2', '5_8S_rRNA', 'U4', 'U6', 'HERC2P7',
       'SNORA16A', 'SNORA17B', '5S_rRNA', 'SNORD22', 'SNORD3D', 'SCARNA4',
       'LINC02203', 'TBCE'],
      dtype='object', name='gene')

In [ ]:
line = ['H1299' if k=='99' else 'H1975' if k=='75' else 'Unknown' for k in df.columns.str[:2]]  # TODO: verify this!
genotype = list(df.columns.str.split('_').str[0].str[2:])
perturbation = list(df.columns.str.split('_').str[1])
guide = list(df.columns.str.split('_').str[2])  # TODO: verify this!
replicate = list(df.columns.str.split('_').str[3])  # TODO: verify this!

In [103]:
coldata = pd.DataFrame({
 "line": line,
 "genotype": genotype,
 "perturbation": perturbation,
 "guide": guide,
 "replicate": replicate
}, index=df.columns)
for index, row in coldata[~coldata.index.str.contains('_V')].iterrows():
    coldata.loc[index, 'guide'] = row.perturbation[-1] if 'MOCK' not in index else None
    coldata.loc[index, 'perturbation'] = row.perturbation[:-1].upper() if 'MOCK' not in index else 'MOCK'
    coldata.loc[index, 'replicate'] = row.guide
coldata.to_csv(DIR / "prelim_combined_coldata.tsv", sep='\t', header=True)

In [104]:
coldata

,line,genotype,perturbation,guide,replicate
75CAS_FBL_V1_R1,H1975,CAS,FBL,V1,R1
75CAS_FBL_V1_R2,H1975,CAS,FBL,V1,R2
75CAS_FBL_V1_R3,H1975,CAS,FBL,V1,R3
75CAS_FBL_V2_R1,H1975,CAS,FBL,V2,R1
75CAS_FBL_V2_R2,H1975,CAS,FBL,V2,R2
...,...,...,...,...,...
99CAS_VIM_V2_R2,H1299,CAS,VIM,V2,R2
99CAS_VIM_V2_R3,H1299,CAS,VIM,V2,R3
99CAS_VIM_V3_R1,H1299,CAS,VIM,V3,R1
99CAS_VIM_V3_R2,H1299,CAS,VIM,V3,R2
